In [ ]:
import numpy as np
import pandas as pd
from ydata_profiling import ProfileReport
import scipy.stats as stats

c:\Users\gg00642\.conda\envs\Env2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
fpath = 'C:\\Users\\gg00642\\OneDrive - University of Surrey\\Desktop\\Actigraphy Sara'

In [4]:
# Read the Excel file
df = pd.read_excel(fpath + '\\5.0_database_variables.xlsx')

In [5]:
df = df.rename(columns={'location(ita=0,uk=1,usa=2)': 'location', 'week(1=free days)': 'weekday_type'})

In [6]:
df = df.rename(columns={'location(ita=0,uk=1,usa=2)': 'location', 'week(1=free days)': 'weekday_type'})

In [ ]:
# list of headers
headers = list(df.columns)

In [8]:
# remove outliers
df = df[(np.abs(stats.zscore(df['sleep_duration(h)_UTC'])) < 3)]
df = df[(np.abs(stats.zscore(df['midpoint_h_UTC'])) < 3)]
df = df[(np.abs(stats.zscore(df['sleep_start_decimal_UTC'])) < 3)]
df = df[(np.abs(stats.zscore(df['sleep_end_decimal_UTC'])) < 3)]
df = df[(np.abs(stats.zscore(df['phase(sleepoffset-sunrise)'])) < 3)]

In [9]:
df = df.drop('sunrise_time(USA)', axis=1)
df = df.drop('sunrise (USA), hours', axis=1)
df = df.drop('sunset_time(USA)', axis=1)
df = df.drop('sunset (USA), hours', axis=1)
df = df.drop('photoperiod (h, USA)', axis=1)

df = df.drop('start_datetime', axis=1)
df = df.drop('end_datetime', axis=1)

df = df.drop('midpoint_datetime', axis=1)
df = df.drop('midpoint_time', axis=1)

df = df.drop('sleep_start_time', axis=1)
df = df.drop('sleep_end_time', axis=1)

df = df.drop('sunrise_time(uk)', axis=1)
df = df.drop('sunset_time(uk)', axis=1)
df = df.drop('sunrise_time(ita)', axis=1)
df = df.drop('sunset_time(ita)', axis=1)

In [10]:
# Define the start date
start_date = pd.to_datetime('2022-09-21')

In [11]:
# Function to calculate the week of the year from the start date
def calculate_week_of_year(date):
    year_diff = date.year - start_date.year
    start_of_year = pd.to_datetime(f'{date.year}-01-01')
    weeks_from_start = ((date - start_of_year).days // 7) + 1
    return year_diff * 52 + weeks_from_start

# Apply the function to calculate the week of the year
df['week_of_year'] = df['date'].apply(calculate_week_of_year)

In [12]:
# adjust 'week of the year' to start from 0
df['week_of_year'] = df['week_of_year'] - 37

In [13]:
# rename the location column as 0=ITA, 1=UK
df['location'] = df['location'].map({0: 'ITA', 1: 'UK'})

# rename the weekday column as 0=work days, 1=free days
df['weekday_type'] = df['weekday_type'].map({0: 'work days', 1: 'free days'})

In [14]:
# if column 'location' = 1 take the value from 'photoperiod (h, UK)' 
# if column 'location' = 0 then photoperiod (h, ITA)'
df['photoperiod'] = np.where(df['location'] == 'UK', df['photoperiod (h, UK)'], df['photoperiod (h, ITA)'])

In [15]:
df = df.drop('photoperiod (h, UK)', axis=1)
df = df.drop('photoperiod (h, ITA)', axis=1)

In [16]:
profile = ProfileReport(df, title="Profiling Report")

In [17]:
# As a file
profile.to_file(fpath + "\\profiling_report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 13.88it/s]
